# ML Zoomcamp 2026 — Homework 2: Regression
Source: https://github.com/DataTalksClub/machine-learning-zoomcamp/blob/main/cohorts/2026/homework/02-regression/homework.md

Predict fuel_efficiency_mpg on its original scale (no log transform). Use the course normal equations, including regularization of the intercept. Run cells in order. The CSV downloads beside this notebook if absent. Requires NumPy and pandas.

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from urllib.request import urlretrieve
import hashlib

DATA_URL = "https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/car_fuel_efficiency_2026.csv"
DATA_PATH = Path("car_fuel_efficiency_2026.csv")
if not DATA_PATH.exists():
    urlretrieve(DATA_URL, DATA_PATH)
print("CSV SHA256:", hashlib.sha256(DATA_PATH.read_bytes()).hexdigest())
features = ["engine_displacement", "horsepower", "vehicle_weight", "model_year"]
target = "fuel_efficiency_mpg"
df = pd.read_csv(DATA_PATH)[features + [target]]
print("Shape:", df.shape)
print("Target quantiles:", df[target].quantile([0, .5, .9, .99, 1]).to_dict())

CSV SHA256: 00a5cab178a8b7cd6e9157ee71dabc236ba7f20b1832336d358b07af14ba431f
Shape: (10000, 5)
Target quantiles: {0.0: 19.8, 0.5: 30.0, 0.9: 33.8, 0.99: 36.9, 1.0: 41.2}


## Q1 and Q2 — Missing values and median
Count missing entries and calculate the median before splitting.

In [2]:
print(df.isna().sum())
missing_column = df.columns[df.isna().any()].item()
median_hp = df.horsepower.median()
print("Q1:", missing_column)
print("Q2:", median_hp)

engine_displacement      0
horsepower             877
vehicle_weight           0
model_year               0
fuel_efficiency_mpg      0
dtype: int64
Q1: horsepower
Q2: 254.0


## Shared functions
The split reproduces the assignment exactly. Lower RMSE is better. The imputation mean must come from training data only.

In [3]:
def split_data(seed):
    n = len(df)
    n_val = int(n * 0.2)
    n_test = int(n * 0.2)
    n_train = n - n_val - n_test
    np.random.seed(seed)
    idx = np.arange(n)
    np.random.shuffle(idx)
    train = df.iloc[idx[:n_train]].copy()
    val = df.iloc[idx[n_train:n_train + n_val]].copy()
    test = df.iloc[idx[n_train + n_val:]].copy()
    return train, val, test

def prepare_X(frame, fill=0):
    return frame[features].fillna(fill).to_numpy()

def train_linear_regression(X, y, r=0):
    X = np.column_stack([np.ones(X.shape[0]), X])
    XTX = X.T.dot(X)
    XTX = XTX + r * np.eye(XTX.shape[0])
    weights = np.linalg.inv(XTX).dot(X.T).dot(y)
    return weights[0], weights[1:]

def rmse(y, pred):
    return np.sqrt(np.mean((y - pred) ** 2))

def evaluate(train, val, fill=0, r=0):
    w0, w = train_linear_regression(prepare_X(train, fill), train[target].to_numpy(), r)
    return float(rmse(val[target].to_numpy(), w0 + prepare_X(val, fill).dot(w)))

train, val, test = split_data(42)
print("Split sizes:", len(train), len(val), len(test))

Split sizes: 6000 2000 2000


## Q3 — Zero vs training mean
Compare validation RMSE rounded to three decimals.

In [4]:
training_mean = train[missing_column].mean()
q3_scores = {"With 0": evaluate(train, val), "With mean": evaluate(train, val, training_mean)}
for option, score in q3_scores.items():
    print(option, score, "rounded:", round(score, 3))
rounded_q3 = {k: round(v, 3) for k, v in q3_scores.items()}
q3 = "Both are equally good" if len(set(rounded_q3.values())) == 1 else min(rounded_q3, key=rounded_q3.get)
print("Q3:", q3)

With 0 2.205293194751098 rounded: 2.205
With mean 2.201817484568115 rounded: 2.202
Q3: With mean


## Q4 — Regularization
Use zero imputation and four-decimal RMSE. Break ties by choosing the smallest r.

In [5]:
r_values = [0, 0.01, 0.1, 1, 5, 10, 100]
q4_scores = {r: evaluate(train, val, r=r) for r in r_values}
for r, score in q4_scores.items():
    print("r =", r, "RMSE =", score, "rounded:", round(score, 4))
q4 = min(r_values, key=lambda r: (round(q4_scores[r], 4), r))
print("Q4:", q4)

r = 0 RMSE = 2.205293194751098 rounded: 2.2053
r = 0.01 RMSE = 2.205827616672006 rounded: 2.2058
r = 0.1 RMSE = 2.2241432777850636 rounded: 2.2241
r = 1 RMSE = 2.349228888392786 rounded: 2.3492
r = 5 RMSE = 2.409380166599679 rounded: 2.4094
r = 10 RMSE = 2.4194698654732387 rounded: 2.4195
r = 100 RMSE = 2.4292021380162367 rounded: 2.4292
Q4: 0


## Q5 — Sensitivity to splitting seed
Use np.std with its default ddof=0, on unrounded RMSE scores. Round only the final standard deviation.

In [6]:
seed_scores = []
for seed in range(10):
    tr, va, te = split_data(seed)
    score = evaluate(tr, va)
    seed_scores.append(score)
    print(seed, score)
q5 = round(float(np.std(seed_scores)), 3)
print("Q5:", q5)

0 2.2392041430461465
1 2.2021128210838907
2 2.163419778753095
3 2.2043588768539144
4 2.2018800943312926
5 2.2457851509084974
6 2.269721207952404
7 2.190794599933433
8 2.2166112016864443
9 2.207036592817851
Q5: 0.029


## Q6 — Final test evaluation
Seed 9; concatenate training and validation; zero imputation; r=0.001. Evaluate on the untouched test set.

In [7]:
tr, va, te = split_data(9)
full_train = pd.concat([tr, va], ignore_index=True)
q6_raw = evaluate(full_train, te, r=0.001)
q6 = round(q6_raw, 3)
print("Test RMSE:", q6_raw)
print("Q6:", q6)
print("ANSWERS:", {"Q1": missing_column, "Q2": median_hp, "Q3": q3, "Q4": q4, "Q5": q5, "Q6": q6})

Test RMSE: 2.2358277471917636
Q6: 2.236
ANSWERS: {'Q1': 'horsepower', 'Q2': np.float64(254.0), 'Q3': 'With mean', 'Q4': 0, 'Q5': 0.029, 'Q6': 2.236}
